In [2]:
import pandas as pd
import numpy as np

# 1. Carga de datos
# Asegurate de tener el archivo listings.csv en la misma carpeta
df = pd.read_csv('listings.csv')
print(f"Registros iniciales: {len(df)}")

# 2. Filtro de columnas relevantes
columnas_utiles = [
    'neighbourhood_cleansed', 'room_type', 'price',
    'review_scores_rating', 'number_of_reviews', 'accommodates'
]
df = df[columnas_utiles]

# 3. Limpieza de formato en la columna 'price' y eliminación de nulos
# Primero borramos los que no tienen precio
df = df.dropna(subset=['price'])

# Luego limpiamos el texto y convertimos a número (float)
df['price'] = df['price'].str.replace('$', '', regex=False).str.replace(',', '', regex=False).astype(float)

# 4. Tratamiento de valores atípicos (Outliers) en precios mediante percentiles 1% y 99%
q_bajo = df['price'].quantile(0.01)
q_alto = df['price'].quantile(0.99)

# Filtramos conservando los que están en ese rango lógico
df = df[(df['price'] >= q_bajo) & (df['price'] <= q_alto)]

# 5. Limpieza de reseñas para asegurar calidad de datos (mínimo 5 reseñas)
# Borramos alojamientos sin puntaje
df = df.dropna(subset=['review_scores_rating'])
# Filtramos alojamientos con historial comprobado
df = df[df['number_of_reviews'] >= 5]

print(f"Registros finales limpios: {len(df)}")

# Exportamos el dataset limpio para usarlo en las siguientes etapas
df.to_csv('listings_limpio.csv', index=False)

Registros iniciales: 29753
Registros finales limpios: 20058


In [ ]:
import pandas as pd

# Cargar el dataset limpio de la etapa anterior
df = pd.read_csv('listings_limpio.csv')

print("--- 1. INDICADORES DE PRECIO ---")
mediana_precio = df['price'].median()
promedio_precio = df['price'].mean()
print(f"Mediana de precio: ${mediana_precio:,.2f}")
print(f"Promedio de precio (solo para comparar sesgo): ${promedio_precio:,.2f}")

print("\n--- 2. PROPORCIONES POR TIPO DE ALOJAMIENTO ---")
proporciones_habitacion = df['room_type'].value_counts(normalize=True) * 100
print(proporciones_habitacion.round(2).astype(str) + '%')

print("\n--- 3. COMPARACIÓN POR BARRIOS (Solo barrios con > 50 avisos) ---")
# Agrupamos calculando mediana de precio, mediana de puntaje y cantidad de avisos
barrios = df.groupby('neighbourhood_cleansed').agg(
    precio_mediana=('price', 'median'),
    puntaje_mediana=('review_scores_rating', 'median'),
    cantidad_avisos=('price', 'count')
).reset_index()

barrios_validos = barrios[barrios['cantidad_avisos'] > 50].copy()

# Top 5 barrios más baratos
barrios_baratos = barrios_validos.sort_values('precio_mediana').head(5)
print("\nTop 5 barrios más accesibles (mediana):")
print(barrios_baratos[['neighbourhood_cleansed', 'precio_mediana', 'puntaje_mediana']])

print("\n--- 4. CORRELACIÓN PRECIO VS PUNTAJE ---")
# Correlación de Spearman (ideal para datos con sesgo o distribuciones no normales)
correlacion = df['price'].corr(df['review_scores_rating'], method='spearman')
print(f"Correlación (Spearman): {correlacion:.4f}")

--- 1. INDICADORES DE PRECIO ---
Mediana de precio: $107,017.25
Promedio de precio (solo para comparar sesgo): $128,528.25

--- 2. PROPORCIONES POR TIPO DE ALOJAMIENTO ---
room_type
Entire home/apt    93.54%
Private room         6.2%
Shared room         0.17%
Hotel room          0.09%
Name: proportion, dtype: object

--- 3. COMPARACIÓN POR BARRIOS (Solo barrios con > 50 avisos) ---

Top 5 barrios más accesibles (mediana):
   neighbourhood_cleansed  precio_mediana  puntaje_mediana
30          San Cristobal       79154.425             4.86
12                 Flores       80569.500             4.89
11           Constitucion       82556.000             4.82
7               Caballito       84849.500             4.88
40          Villa Ortuzar       84907.650             4.90

--- 4. CORRELACIÓN PRECIO VS PUNTAJE ---
Correlación (Spearman): 0.0834


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración inicial para gráficos bonitos
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Cargar dataset limpio
df = pd.read_csv('listings_limpio.csv')

# Preparamos datos auxiliares
barrios_agrupados = df.groupby('neighbourhood_cleansed').agg(
    mediana_precio=('price', 'median'),
    cantidad=('price', 'count')
).reset_index()
barrios_validos = barrios_agrupados[barrios_agrupados['cantidad'] > 50]
top_10_volumen = barrios_validos.sort_values('cantidad', ascending=False).head(10)['neighbourhood_cleansed']
df_top10 = df[df['neighbourhood_cleansed'].isin(top_10_volumen)]


# ==========================================
# GRÁFICO 1: Histograma (Distribución de precios)
# ==========================================
plt.figure()
sns.histplot(df['price'], bins=50, kde=True, color='skyblue')
plt.title('G1: Distribución del Precio por Noche (Todo CABA)')
plt.xlabel('Precio por Noche (ARS)')
plt.ylabel('Cantidad de Alojamientos')
plt.xlim(0, 400000) # Recortamos el eje X para mejor legibilidad
plt.tight_layout()
plt.savefig('grafico_1_histograma.png')
plt.close()

# ==========================================
# GRÁFICO 2: Gráfico de Barras Horizontal (Precios por Barrio)
# ==========================================
barrios_baratos = barrios_validos.sort_values('mediana_precio').head(10)
plt.figure()
sns.barplot(data=barrios_baratos, x='mediana_precio', y='neighbourhood_cleansed', palette='viridis')
plt.title('G2: Top 10 Barrios más accesibles (Mediana de Precio)')
plt.xlabel('Precio Mediano (ARS)')
plt.ylabel('Barrio')
plt.tight_layout()
plt.savefig('grafico_2_barras_horizontales.png')
plt.close()

# ==========================================
# GRÁFICO 3: Gráfico de Dispersión / Scatter (Precio vs Puntaje)
# ==========================================
plt.figure()
sns.scatterplot(data=df, x='price', y='review_scores_rating', alpha=0.3, color='coral')
plt.title('G3: Relación entre Precio por Noche y Puntaje de Reseñas')
plt.xlabel('Precio por Noche (ARS)')
plt.ylabel('Puntaje General (1 a 5)')
plt.xlim(0, 400000)
plt.tight_layout()
plt.savefig('grafico_3_dispersion.png')
plt.close()

# ==========================================
# GRÁFICO 4: Boxplot (Dispersión de precios por tipo de habitación)
# ==========================================
plt.figure()
sns.boxplot(data=df, x='room_type', y='price', palette='Set2')
plt.title('G4: Dispersión de Precios según Tipo de Alojamiento')
plt.xlabel('Tipo de Alojamiento')
plt.ylabel('Precio por Noche (ARS)')
plt.ylim(0, 300000) # Limitamos Y para ver bien las cajas
plt.tight_layout()
plt.savefig('grafico_4_boxplot.png')
plt.close()

# ==========================================
# GRÁFICO 5: Gráfico de Barras Verticales (Volumen de oferta)
# ==========================================
oferta_barrios = barrios_validos.sort_values('cantidad', ascending=False).head(10)
plt.figure()
sns.barplot(data=oferta_barrios, x='neighbourhood_cleansed', y='cantidad', palette='mako')
plt.title('G5: Top 10 Barrios con mayor cantidad de anuncios')
plt.xlabel('Barrio')
plt.ylabel('Cantidad de Alojamientos')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('grafico_5_barras_verticales.png')
plt.close()

# ==========================================
# GRÁFICO 6: Mapa de Calor / Heatmap (Precio por Barrio y Capacidad)
# ==========================================
# Filtramos alojamientos para 1 a 4 personas en los 10 barrios principales
df_heatmap = df[df['neighbourhood_cleansed'].isin(top_10_volumen) & df['accommodates'].isin([1, 2, 3, 4])]
pivot_heatmap = pd.pivot_table(df_heatmap, values='price', index='neighbourhood_cleansed', columns='accommodates', aggfunc='median')

plt.figure()
sns.heatmap(pivot_heatmap, annot=True, fmt=".0f", cmap='YlGnBu', cbar_kws={'label': 'Precio Mediano (ARS)'})
plt.title('G6: Mediana de Precio por Barrio según Capacidad (1 a 4 huéspedes)')
plt.xlabel('Capacidad (Personas)')
plt.ylabel('Barrio')
plt.tight_layout()
plt.savefig('grafico_6_heatmap.png')
plt.close()

# ==========================================
# GRÁFICO 7: Gráfico de Violín (Distribución de puntajes por Tipo)
# ==========================================
# Filtramos solo Private room y Entire home (99% de la muestra)
df_violin = df[df['room_type'].isin(['Entire home/apt', 'Private room'])]
plt.figure()
sns.violinplot(data=df_violin, x='room_type', y='review_scores_rating', palette='pastel')
plt.title('G7: Distribución de Puntajes por Tipo de Alojamiento')
plt.xlabel('Tipo de Alojamiento')
plt.ylabel('Puntaje General')
plt.tight_layout()
plt.savefig('grafico_7_violin.png')
plt.close()

# ==========================================
# GRÁFICO 8: Barras Apiladas (Composición 100% por Barrio)
# ==========================================
# Proporción de tipo de habitación en los top 5 barrios
top_5_barrios = oferta_barrios.head(5)['neighbourhood_cleansed']
df_barras_apiladas = df[df['neighbourhood_cleansed'].isin(top_5_barrios)]
crosstab_room = pd.crosstab(df_barras_apiladas['neighbourhood_cleansed'], df_barras_apiladas['room_type'], normalize='index') * 100

plt.figure()
crosstab_room.plot(kind='bar', stacked=True, colormap='Set3', figsize=(10, 6))
plt.title('G8: Proporción del Tipo de Alojamiento en Top 5 Barrios')
plt.xlabel('Barrio')
plt.ylabel('Porcentaje (%)')
plt.legend(title='Tipo de Habitación', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('grafico_8_barras_apiladas.png')
plt.close()

/tmp/ipykernel_1699/3192490549.py:41: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=barrios_baratos, x='mediana_precio', y='neighbourhood_cleansed', palette='viridis')
/tmp/ipykernel_1699/3192490549.py:66: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df, x='room_type', y='price', palette='Set2')
/tmp/ipykernel_1699/3192490549.py:80: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=oferta_barrios, x='neighbourhood_cleansed', y='cantidad', palette='mako')
/tmp/ipykernel_1699/3192490549.py:111: FutureWarning: 

Passing `palette` wi

<Figure size 1000x600 with 0 Axes>

In [3]:
import pandas as pd
from scipy import stats

df = pd.read_csv('listings_limpio.csv')
B = 'neighbourhood_cleansed'
mediana_ciudad = df['price'].median()
puntaje_mediano = df['review_scores_rating'].median()

print("=== Dataset ===")
print("Registros:", len(df), "| Nulos:", int(df.isna().sum().sum()), "| Duplicados:", int(df.duplicated().sum()))
print("Barrios:", df[B].nunique(), "| Precio min/max:", df['price'].min(), df['price'].max())
print("Puntaje min/max:", df['review_scores_rating'].min(), df['review_scores_rating'].max())
print("Capacidad min/max:", df['accommodates'].min(), df['accommodates'].max())

print("\n=== Precio: tendencia central y dispersión ===")
q1, q3 = df['price'].quantile([.25, .75])
print(f"Mediana ${mediana_ciudad:,.0f} | Promedio ${df['price'].mean():,.0f} | Q1 ${q1:,.0f} | Q3 ${q3:,.0f} | IQR ${q3-q1:,.0f}")
print(f"Máximo / mediana = {df['price'].max()/mediana_ciudad:.1f}x")
print(f"% de avisos < $120.000: {(df['price'] < 120000).mean()*100:.1f}%")
print("IQR por tipo:")
for t, g in df.groupby('room_type')['price']:
    a, b = g.quantile([.25, .75])
    print(f"  {t}: n={len(g)}, mediana ${g.median():,.0f}, IQR ${b-a:,.0f}")

print("\n=== Composición por tipo (%) ===")
print((df['room_type'].value_counts(normalize=True) * 100).round(2))

print("\n=== Gráficos truncados (avisos que NO se ven) ===")
for lim in (300000, 400000):
    print(f"Precio > ${lim:,}: {(df['price'] > lim).sum()} avisos ({(df['price'] > lim).mean()*100:.1f}%)")

print("\n=== P1: barrios (>50 avisos) con precio < mediana ciudad y puntaje >= mediana ===")
g = df.groupby(B).agg(n=('price', 'size'), precio=('price', 'median'),
                      puntaje=('review_scores_rating', 'median')).query('n > 50')
pal = g.loc['Palermo', 'precio']
g['ahorro_vs_ciudad_%'] = ((1 - g['precio'] / mediana_ciudad) * 100).round(1)
g['ahorro_vs_Palermo_%'] = ((1 - g['precio'] / pal) * 100).round(1)
sel = g[(g['precio'] < mediana_ciudad) & (g['puntaje'] >= puntaje_mediano)].sort_values('precio')
print(sel)
print("Rango de puntajes entre barrios >50:", g['puntaje'].min(), "a", g['puntaje'].max())

print("\n=== P2: Spearman precio vs puntaje ===")
rho, p = stats.spearmanr(df['price'], df['review_scores_rating'])
print(f"rho = {rho:.3f} | p = {p:.2e} | rho^2 = {rho**2*100:.2f}%")
print(df.groupby(pd.qcut(df['price'], 4))['review_scores_rating'].median())
print(f"% de avisos con puntaje >= 4.5: {(df['review_scores_rating'] >= 4.5).mean()*100:.1f}%")

print("\n=== P3: departamento entero vs habitación privada ===")
caros = g.sort_values('precio', ascending=False).head(5).index.tolist()
print("5 barrios más caros (mediana, >50 avisos):", caros)
s = df[df[B].isin(caros)]
apt = s[s.room_type == 'Entire home/apt']['price']
priv = s[s.room_type == 'Private room']['price']
print(f"Depto: n={len(apt)}, mediana ${apt.median():,.0f} | Privada: n={len(priv)}, mediana ${priv.median():,.0f}")
print(f"Ahorro: ${apt.median()-priv.median():,.0f} ({(1-priv.median()/apt.median())*100:.1f}%)")
print("Capacidad mediana por tipo:", s.groupby('room_type')['accommodates'].median().to_dict())
print("Comparación a igual capacidad:")
for cap in (1, 2, 3, 4):
    x = s[s.accommodates == cap]
    a, p_ = x[x.room_type == 'Entire home/apt']['price'], x[x.room_type == 'Private room']['price']
    ahorro = f"{(1-p_.median()/a.median())*100:.1f}%" if len(p_) else "s/d"
    print(f"  {cap} huésp.: depto n={len(a)} ${a.median():,.0f} | privada n={len(p_)} ${p_.median():,.0f} | ahorro {ahorro}")

print("\n=== G7: puntajes por tipo ===")
for t in ('Entire home/apt', 'Private room'):
    x = df[df.room_type == t]['review_scores_rating']
    print(f"{t}: mediana {x.median()}, media {x.mean():.2f}, % con 5.0 = {(x == 5).mean()*100:.1f}%")
u = stats.mannwhitneyu(df[df.room_type == 'Entire home/apt']['review_scores_rating'],
                       df[df.room_type == 'Private room']['review_scores_rating'])
print(f"Mann-Whitney p = {u.pvalue:.2f}")

print("\n=== G8: % de departamentos enteros en los 5 barrios con más oferta ===")
top5 = df[B].value_counts().head(5).index
print((pd.crosstab(df[df[B].isin(top5)][B], df['room_type'], normalize='index') * 100).round(1))

=== Dataset ===
Registros: 20058 | Nulos: 0 | Duplicados: 31
Barrios: 46 | Precio min/max: 27000.0 895121.0
Puntaje min/max: 2.4 5.0
Capacidad min/max: 1 16

=== Precio: tendencia central y dispersión ===
Mediana $107,017 | Promedio $128,528 | Q1 $82,351 | Q3 $145,238 | IQR $62,887
Máximo / mediana = 8.4x
% de avisos < $120.000: 61.0%
IQR por tipo:
  Entire home/apt: n=18763, mediana $109,310, IQR $62,849
  Hotel room: n=18, mediana $117,184, IQR $163,294
  Private room: n=1243, mediana $53,509, IQR $36,692
  Shared room: n=34, mediana $31,956, IQR $10,620

=== Composición por tipo (%) ===
room_type
Entire home/apt    93.54
Private room        6.20
Shared room         0.17
Hotel room          0.09
Name: proportion, dtype: float64

=== Gráficos truncados (avisos que NO se ven) ===
Precio > $300,000: 807 avisos (4.0%)
Precio > $400,000: 395 avisos (2.0%)

=== P1: barrios (>50 avisos) con precio < mediana ciudad y puntaje >= mediana ===
                          n      precio  puntaje  ah

/tmp/ipykernel_505/1623911089.py:45: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby(pd.qcut(df['price'], 4))['review_scores_rating'].median())
